In [1]:
import pandas as pd
import yfinance as yf
import datetime as dt

In [2]:
start=dt.datetime(2020,1,1)
end=dt.datetime(2023,12,31)
stk_data=yf.download("RELIANCE.NS",start="2020-1-1",end="2023-12-31")

[*********************100%***********************]  1 of 1 completed


In [3]:
stk_data.columns=stk_data.columns.get_level_values(0)

In [4]:
stk_data=stk_data[["Open","High","Low","Close"]]

In [5]:
data1=pd.DataFrame(stk_data,columns=["Open","High","Low","Close"])

In [6]:
data1

,Open,High,Low,Close
Date,,,,
2020-01-01,675.956671,680.008852,670.390532,672.216187
2020-01-02,673.284861,686.176147,673.284861,683.660217
2020-01-03,682.636062,686.487896,678.183128,684.484070
2020-01-06,676.847260,680.365044,667.050769,668.609314
2020-01-07,676.401995,683.304098,673.952887,678.895691
...,...,...,...,...
2023-12-22,1264.550414,1275.073416,1258.646514,1267.242920
2023-12-26,1268.700379,1280.532666,1266.081931,1273.665527
2023-12-27,1275.617031,1284.460338,1271.220103,1278.013184


In [7]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

794
X_train length: (794, 4)
X_test length: (198, 4)
y_train length: (794, 4)
y_test length: (198, 4)


In [8]:
import warnings
warnings.filterwarnings("ignore")

In [9]:
performance={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [10]:
listt=["Close","High","Open","Low"]

In [11]:
datasetTwo=data1[listt]
datasetTwo

,Close,High,Open,Low
Date,,,,
2020-01-01,672.216187,680.008852,675.956671,670.390532
2020-01-02,683.660217,686.176147,673.284861,673.284861
2020-01-03,684.484070,686.487896,682.636062,678.183128
2020-01-06,668.609314,680.365044,676.847260,667.050769
2020-01-07,678.895691,683.304098,676.401995,673.952887
...,...,...,...,...
2023-12-22,1267.242920,1275.073416,1264.550414,1258.646514
2023-12-26,1273.665527,1280.532666,1268.700379,1266.081931
2023-12-27,1278.013184,1284.460338,1275.617031,1271.220103


In [12]:
test_obs=28
train =datasetTwo[:-test_obs]  #-test_obs=select all ros except the last 28 rows 
test = datasetTwo[-test_obs:]  # select the last 28 rows

In [13]:
from sklearn.preprocessing import MinMaxScaler
ms = MinMaxScaler()
train_scaled=ms.fit_transform(train)
test_scaled=ms.transform(test)

In [14]:
train_scaled

array([[0.31031032, 0.29378309, 0.30258711, 0.3183443 ],
       [0.32305902, 0.30083364, 0.29957024, 0.32162949],
       [0.3239768 , 0.30119004, 0.31012913, 0.32718924],
       ...,
       [0.85836689, 0.85040871, 0.8446938 , 0.86230941],
       [0.86070587, 0.85729913, 0.85088587, 0.87349646],
       [0.85787163, 0.85679096, 0.85188998, 0.87635648]], shape=(964, 4))

In [15]:
from statsmodels.tsa.api import VAR
for i in [1,2,3,4,5,6,7,8,9,10]:
    model=VAR(train_scaled)
    results=model.fit(i)
    print("Order=",i)
    print("AIC:",results.aic)
    print("BIC:",results.bic)

Order= 1
AIC: -37.5449696231026
BIC: -37.44382625006422
Order= 2
AIC: -37.57364778859105
BIC: -37.3914393476098
Order= 3
AIC: -37.596289485286796
BIC: -37.33288192101369
Order= 4
AIC: -37.59300483481351
BIC: -37.24826372716413
Order= 5
AIC: -37.58824478887148
BIC: -37.16203535165541
Order= 6
AIC: -37.569550324507766
BIC: -37.06173740405152
Order= 7
AIC: -37.554086717819736
BIC: -36.96453479158301
Order= 8
AIC: -37.543709597480735
BIC: -36.872282772666104
Order= 9
AIC: -37.5225477224281
BIC: -36.769109734583964
Order= 10
AIC: -37.501529325330935
BIC: -36.66594353694779


In [16]:
x = model.select_order(maxlags=12)  #select_order() checks different lag orders.
#maxlags=12 means it tests lag values from 0 to 12.

In [17]:
x.selected_orders

{'aic': np.int64(3),
 'bic': np.int64(1),
 'hqic': np.int64(1),
 'fpe': np.int64(3)}

In [18]:
order=x.selected_orders["aic"] #"aic" retrieves the lag with the lowest AIC.
order

np.int64(3)

In [19]:
result = model.fit(order)
result

In [20]:
lagged_values=train_scaled[-order:]
lagged_values  #select the last 3 rows here order =3

array([[0.85836689, 0.85040871, 0.8446938 , 0.86230941],
       [0.86070587, 0.85729913, 0.85088587, 0.87349646],
       [0.85787163, 0.85679096, 0.85188998, 0.87635648]])

In [21]:
pred=result.forecast(y=lagged_values,steps=28)
pred

#y=lagged_Values → the last order observations from the training data.
#steps=28 → forecast the next 28 time periods.

array([[0.85762478, 0.8585739 , 0.85501793, 0.86779667],
       [0.85532765, 0.85653649, 0.85375087, 0.86594561],
       [0.85359117, 0.85479773, 0.85164901, 0.86399194],
       [0.85323227, 0.85453214, 0.85000436, 0.86240265],
       [0.85299445, 0.85435292, 0.84971563, 0.86200661],
       [0.85276267, 0.85428406, 0.84953166, 0.86165723],
       [0.85241844, 0.85415583, 0.84924787, 0.86124772],
       [0.85208483, 0.85388856, 0.84890517, 0.86084491],
       [0.85174817, 0.85361368, 0.84856659, 0.86044927],
       [0.85139423, 0.85330787, 0.84822116, 0.86005743],
       [0.85103783, 0.85297701, 0.84786216, 0.85967025],
       [0.85067906, 0.85263599, 0.84750023, 0.85928659],
       [0.8503194 , 0.85228588, 0.84713681, 0.85890687],
       [0.84995963, 0.85193029, 0.84677276, 0.85853077],
       [0.84960016, 0.85157143, 0.84640871, 0.85815745],
       [0.8492416 , 0.85121061, 0.84604516, 0.8577868 ],
       [0.84888425, 0.85084907, 0.84568265, 0.85741868],
       [0.84852834, 0.85048758,

In [22]:
preds=pd.DataFrame(pred,columns=listt)

In [23]:
preds

,Close,High,Open,Low
0,0.857625,0.858574,0.855018,0.867797
1,0.855328,0.856536,0.853751,0.865946
2,0.853591,0.854798,0.851649,0.863992
3,0.853232,0.854532,0.850004,0.862403
4,0.852994,0.854353,0.849716,0.862007
5,0.852763,0.854284,0.849532,0.861657
6,0.852418,0.854156,0.849248,0.861248
7,0.852085,0.853889,0.848905,0.860845
8,0.851748,0.853614,0.848567,0.860449
9,0.851394,0.853308,0.848221,0.860057


In [24]:
preds.to_csv("varforecasted_{}.csv".format(test_obs))

In [25]:
import numpy as np
from sklearn.metrics import mean_squared_error
rmse=round(np.sqrt(mean_squared_error(test_scaled,pred)))

In [26]:
rmse

0

In [27]:
from sklearn.metrics import mean_absolute_percentage_error
mape=mean_absolute_percentage_error(test_scaled,pred)

In [28]:
mape

0.07320199769253063

In [29]:
performance["Model"].append(listt)
performance["RMSE"].append(rmse)
performance["MaPe"].append(mape)
performance["Lag"].append(order)
performance["Test"].append(test_obs)
perf=pd.DataFrame(performance)

In [30]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High, Open, Low]",0,0.073202,3,28


In [34]:
def combination(dataset,listt):
    print(listt)
    datasetTwo=dataset[listt]
    test_obs=28
    train =datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]
    train_scaled=ms.fit_transform(train)
    test_scaled=ms.transform(test)
    from statsmodels.tsa.api import VAR
    for i in [1,2,3,4,5,6,7,8,9,10]:
        model=VAR(train_scaled)
        results=model.fit(i)
        print("Order=",i)
        print("AIC:",results.aic)
        print("BIC:",results.bic)
        print()
    x=model.select_order(maxlags=12)
    order=x.selected_orders["aic"]
    result=model.fit(order)

    lagged_values=train_scaled[-order:]
    pred=result.forecast(y=lagged_values,steps=28)
    preds=pd.DataFrame(pred,columns=listt)
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
    from sklearn.metrics import mean_squared_error
    rmse= round(mean_squared_error(test_scaled,pred))
    from sklearn.metrics import mean_absolute_percentage_error
    mape=mean_absolute_percentage_error(test_scaled,pred)
    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append(order)
    performance["Test"].append(test_obs)
    perf=pd.DataFrame(performance)
    return perf,result,pred
    
    

In [35]:
perf,result,pred=combination(data1,listt)

['Close', 'High', 'Open', 'Low']
Order= 1
AIC: -37.5449696231026
BIC: -37.44382625006422

Order= 2
AIC: -37.57364778859105
BIC: -37.3914393476098

Order= 3
AIC: -37.596289485286796
BIC: -37.33288192101369

Order= 4
AIC: -37.59300483481351
BIC: -37.24826372716413

Order= 5
AIC: -37.58824478887148
BIC: -37.16203535165541

Order= 6
AIC: -37.569550324507766
BIC: -37.06173740405152

Order= 7
AIC: -37.554086717819736
BIC: -36.96453479158301

Order= 8
AIC: -37.543709597480735
BIC: -36.872282772666104

Order= 9
AIC: -37.5225477224281
BIC: -36.769109734583964

Order= 10
AIC: -37.501529325330935
BIC: -36.66594353694779



In [36]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High, Open, Low]",0,0.073202,3,28
1,"[Close, High, Open, Low]",0,0.073202,3,28
2,"[Close, High, Open, Low]",0,0.073202,3,28
